In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from src.utils.preprocessing import wrangle_data
from sklearn.preprocessing import FunctionTransformer
from src.models.evaluate import evaluate, evaluate_anomaly, print_final_results, identify_best_model, find_anomaly_optimal_threshold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import IsolationForest, RandomForestClassifier

In [ ]:
# wrangle data
df = wrangle_data(False)

In [ ]:
# prepare features and target
X = df.drop(columns=["IS_FRAUD"])
y = df["IS_FRAUD"]

In [ ]:
#split data into train, validation and test using temporal split
cutoff = int(len(X) * 0.8)
X_train_full, y_train_full = X.iloc[: cutoff], y.iloc[:cutoff]
X_test, y_test =  X.iloc[cutoff: ], y.iloc[cutoff:]

cutoff_train_val = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full.iloc[:cutoff_train_val], y_train_full.iloc[:cutoff_train_val]
X_validation, y_validation = X_train_full.iloc[cutoff_train_val:], y_train_full.iloc[cutoff_train_val:]


In [ ]:
#X = 1048575
print("X length " + str(len(X)))
print("X_train length " +str(len(X_train)) + ", y_train length " +str(len(y_train)))
print("X_val length " +str(len(X_validation)) + ", y_val length " +str(len(y_validation)))
print("X_test length " +str(len(X_test))  + ", y_test length " +str(len(y_test)))

In [ ]:
random_forest_pipeline = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])
random_forest_pipeline.fit(X_train, y_train)

In [ ]:
# train isolation forest
CONTAMINATION = 0.0013  # observed fraud rate of 0.13%
iso_model = IsolationForest(contamination=CONTAMINATION, random_state=42, n_jobs=-1)
iso_model.fit(X_train)

In [ ]:
random_forest_test_probs = random_forest_pipeline.predict_proba(X_test)

In [ ]:
# Invert the anomaly scores for training data to make them more intuitive (higher score = more anomalous)
inverted_iso_model_test_score = -iso_model.decision_function(X_test)

In [ ]:
inverted_iso_model_test_score[:10]

In [ ]:
random_forest_best_threshold = 0.4000 # as seen in the shap.ipynb

In [ ]:
original_score_threshold = find_anomaly_optimal_threshold(y_test, inverted_iso_model_test_score)
print(original_score_threshold)

In [ ]:
#model prection for 0  and 1 for each row and shap value for the the two classes for each row
comparison_df = pd.DataFrame({
    'Random_Forest_Prediction': (random_forest_test_probs[:, 1] >= random_forest_best_threshold).astype(int),
    'Isolation_Forest_Prediction': (inverted_iso_model_test_score >= original_score_threshold).astype(int),
    'Actual_Class': y_test
}).reset_index(drop=True)

In [ ]:
comparison_df.head(50)

In [ ]:
rf_misses = comparison_df[(comparison_df['Actual_Class'] == 1) & (comparison_df['Random_Forest_Prediction'] == 0)]

In [ ]:
rf_misses.head(10)
print(len(rf_misses))

In [ ]:
iso_misses = comparison_df[(comparison_df['Actual_Class'] == 1) & (comparison_df['Isolation_Forest_Prediction'] == 0)]

In [ ]:
iso_misses.loc[int(117677)]
print(len(iso_misses))

In [ ]:
rf_misses['Isolation_Forest_Prediction'].mean()   # fraction of RF's misses that IF also flagged

In [ ]:
iso_misses['Random_Forest_Prediction'].mean()   # fraction of Isolation Forest's misses that Random Forest also flagged